<a href="https://colab.research.google.com/github/hiiamlars/master_thesis/blob/main/open_review_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [1]:
# @title Dependencies

# Installations of third-party libraries
!pip install requests -q
!pip install tqdm -q
!pip install beautifulsoup4 -q

# First-party imports
import os
import json
import csv
import random
import argparse
import unicodedata
import threading
import time
import sys
import zipfile
from pathlib import Path
import xml.etree.ElementTree as ET

# Third-party imports
import requests
import pandas as pd
from tqdm.notebook import tqdm
from bs4 import BeautifulSoup
from google.colab import drive

print("\nInstalling and loading dependencies complete!")


Installing and loading dependencies complete!


In [ ]:
# @title Paths

# Mount GDrive
drive.mount('/content/drive')

# Set working directory
WORKING_DIR = "/content/drive/MyDrive/Thesis/iclr/"

# Create working directory (if not already existing)
os.makedirs(WORKING_DIR, exist_ok=True)
print(f"Ensured working directory exists at: {WORKING_DIR}.")

# Define and create RAW_DIR for raw JSONs
RAW_DIR = os.path.join(WORKING_DIR, "raw")
os.makedirs(RAW_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {RAW_DIR}.")

# Define and create SAMPLE_DIR for raw and sampled JSONs
SAMPLE_DIR = os.path.join(WORKING_DIR, "sample")
os.makedirs(SAMPLE_DIR, exist_ok=True)
print(f"Ensured sample directory exists at: {SAMPLE_DIR}.")

# Define and create PDF_DIR for downloaded PDFs
PDF_DIR = os.path.join(WORKING_DIR, "pdf")
os.makedirs(PDF_DIR, exist_ok=True)
print(f"Ensured PDF directory exists at: {PDF_DIR}.")

# Define and create GROBID_DIR for TEI XML files
GROBID_DIR = os.path.join(WORKING_DIR, "grobid")
os.makedirs(GROBID_DIR, exist_ok=True)
print(f"Ensured GROBID output directory exists at: {GROBID_DIR}.")

# Define and create TXT_DIR for extracted text files
TXT_DIR = os.path.join(WORKING_DIR, "txt")
os.makedirs(TXT_DIR, exist_ok=True)
print(f"Ensured TXT output directory exists at: {TXT_DIR}.")

# Define and create LANDINGAI_DIR for LandingAI JSON files
LANDINGAI_DIR = os.path.join(WORKING_DIR, "landingai")
os.makedirs(LANDINGAI_DIR, exist_ok=True)
print(f"Ensured LandingAI output directory exists at: {LANDINGAI_DIR}.")

# Define and create DATASET_DIR for the resulting data file
DATASET_DIR = os.path.join(WORKING_DIR, "final")
os.makedirs(DATASET_DIR, exist_ok=True)
print(f"Ensured dataset directory exists at: {DATASET_DIR}.")

print("\nDefining paths complete!")

In [ ]:
# @title Definitions

# Configuration for Sampling
MAX_PAPERS_TO_SAMPLE = 2000 # Max number of papers to include in the sample
DATASET_NAME_PREFIX = "rand" # Prefix for the sampled dataset name

print("\nDefine sampling configuration complete!")

In [ ]:
# @title APIs

# OpenReviewAPI
OPEN_REVIEW_API = "https://api.openreview.net/notes?invitation=ICLR.cc/2023/Conference/-/Blind_Submission&details=directReplies"

# GrobidAPI
GROBID_API = "https://kermitt2-grobid.hf.space/api/processFulltextDocument"

# LandingAIAPI
LANDING_AI_API = 'https://api.va.landing.ai/v1/ade/parse'

## LandingAIAPI-Key
LANDING_AI_API_KEY = "API-Key place holder"

## LandingAI-Header
LANDING_AI_HEADERS = {'Authorization': f'Bearer {LANDING_AI_API_KEY}'}

## LandingAI-Model
LANDING_AI_MODEL = {'model': 'dpt-2-latest'}

print("\nAPI setup complete!")

# ICLR Data

In [ ]:
# @title Crawl raw data

def get_iclr_reviews(paper_info_filename=None, review_info_filename=None, decision_info_filename=None):
    """Requests, checks and stores ICLR 2023 reviews via the API."""

    offset = 0
    limit = 100 # Number of paper per request
    max_limit = float('inf') # Maximal amount of requests
    paper_list = [] # Create empty paper list
    review_list = [] # Create empty review list

    while offset + limit <= max_limit:
        print(f'Scraping review data: {offset}/{max_limit}') # Print progress
        print("Start request")
        response = requests.get(OPEN_REVIEW_API, params={"offset": offset, "limit": limit}) # Save responses
        print("End request")
        if response.status_code != 200: # Check for warning messages
            print("Error:", response.status_code)
            break

        data = response.json()
        if not data or 'notes' not in data or not data['notes']: # Check content of response
            break

        for note in data['notes']:
            paper = {} # Create empty storage for the following paper information

            paper['uid'] = note['id']
            paper['number'] = note['number']
            paper['title'] = note['content']['title']
            paper['authors'] = note['content']['authors']
            paper['abstract'] = note['content']['abstract']
            paper['pdf'] = note['content']['pdf']
            paper['keywords'] = note['content']['keywords']

            if 'details' in note and 'directReplies' in note['details']: # Check for complete information
                paper_list.append(paper) # Only then append paper information to the empty list
                for directReplies in note['details']['directReplies']:

                    if directReplies['invitation'].endswith("Official_Review"): # Check for reply information

                        review = {}
                        review['uid'] = directReplies['id']
                        review['paper_uid'] = paper['uid']
                        review['paper_title'] = paper['title']
                        review.update(directReplies['content'])

                        review_list.append(review) # Only then append the paper information to the empty list

        offset += limit # Iterate

    with open(paper_info_filename, 'w', encoding = "utf-8") as f:
        json.dump(paper_list, f) # Write paper_info-file
    with open(review_info_filename, 'w', encoding = "utf-8") as f:
        json.dump(review_list, f) # Write review_info-file

if __name__ == "__main__":
    get_iclr_reviews(paper_info_filename = RAW_DIR + "ICLR2023paper_raw.json", # Create 'paper_info_filename'
                     review_info_filename = RAW_DIR + "ICLR2023review_raw.json") # Create 'review_info_filename'

    print("Finished scraping ICLR 2023 reviews!")

In [ ]:
# @title Sampling

def to_ascii(input_str):  # Change the utf-8 characters to ascii
    """Converts a string to ASCII, removing non-ASCII characters."""

    assert(type(input_str) == str) # Assert string input
    normalized = unicodedata.normalize("NFKD", input_str) # Unicode normalization
    normalized = normalized.replace('"', "'")  # Always use ' instead of " to avoid error in json
    ascii_str = "".join(c for c in normalized if c.isascii()) # Drops non-ascii characters
    return ascii_str

random.seed(19260817) # Create parser
parser = argparse.ArgumentParser()
parser.add_argument("--max_count", type=int, default=10**9) # States which arguments are expected
# Update default input paths to SAMPLE_DIR
parser.add_argument("--input_paper", type=str, default=os.path.join(SAMPLE_DIR, "ICLR2023paper_raw.json"))
parser.add_argument("--input_review", type=str, default=os.path.join(SAMPLE_DIR, "ICLR2023review_raw.json"))
parser.add_argument("--dataset_name", type=str, required=True)

# Use predefined variables for sampling configuration
sampled_dataset_name = f"{DATASET_NAME_PREFIX}{MAX_PAPERS_TO_SAMPLE}"
args = parser.parse_args(["--max_count", str(MAX_PAPERS_TO_SAMPLE), "--dataset_name", sampled_dataset_name]) # Select up to MAX_PAPERS_TO_SAMPLE papers and store them

# Read businesses in full dataset, and then rename "uid" key to "id"
with open(args.input_paper, mode="r", encoding="utf-8") as json_file:
    papers = json.load(json_file)
review_cnt = {} # Amount of reviews per paper
for paper in papers:
    paper["id"] = paper["uid"]  # Rename "uid" -> "id"
    paper.pop("uid", None) # Delete "uid"
    paper["title"] = to_ascii(str(paper["title"]))
    paper["keywords"] = to_ascii(str(paper["keywords"]))
    paper["abstract"] = to_ascii(str(paper["abstract"]))
    review_cnt[paper["id"]] = 0 # Initialze count at 0

# Read reviews in full dataset, and then rename "text" key to "review" and rename "business_id" key to "belong_id"
with open(args.input_review, mode="r", encoding="utf-8") as json_file:
    reviews = json.load(json_file)

# Create review format string
review_format_str = '''Summary Of The Paper:

{}

Strength And Weaknesses:

{}

Clarity, Quality, Novelty And Reproducibility:

{}

Summary Of The Review:

{}
'''

for review in reviews:
    review["belong_id"] = review["paper_uid"]  # Rename "paper_uid" -> "belong_id"
    review.pop("paper_uid", None) # Delete "paper_uid"
    review["review"] = review_format_str.format(review["summary_of_the_paper"],review["strength_and_weaknesses"],review["clarity,_quality,_novelty_and_reproducibility"],review["summary_of_the_review"]) # Apply review format string
    review_cnt[review["belong_id"]] += 1 # Increment review count of the paper

paper_selected = []
has = {}
for paper in papers:

    def legal(paper):
        if review_cnt[paper["id"]] < 3: return False # Only consider papers with at least three reviews
        return True

    if legal(paper):

        paper_selected.append(paper) # Select paper
        has[paper["id"]] = [] # Create yet empty id-list

# Randomizes and limits the amount of papers selected
random.shuffle(paper_selected)
if len(paper_selected) > args.max_count:
    paper_selected = paper_selected[:args.max_count]

print("# items =", len(paper_selected))
with open(os.path.join(SAMPLE_DIR, f"paper_{args.dataset_name}.json"),"w",encoding="utf-8") as json_file: # Create a json-file with the paper information
    json.dump(paper_selected, json_file)

for i, review in enumerate(reviews):
    if review["belong_id"] in has: # Only reviews belonging to a selected paper
        has[review["belong_id"]].append(i) # Stores it's current review index i
review_selected = []
for paper in paper_selected:
    for i in has[paper["id"]]:
        reviews[i]["review"] = to_ascii(reviews[i]["review"])
        review_selected.append(reviews[i]) # Adds reviews to selected list

print("# reviews =", len(review_selected))
with open(os.path.join(SAMPLE_DIR, f"review_{args.dataset_name}.json"),"w",encoding="utf-8") as json_file: # Create a json-file with the reviews information
    json.dump(review_selected, json_file)

print("\nSampling complete!")

# Original Papers


In [ ]:
# @title Download PDFs

with open(os.path.join(SAMPLE_DIR, f"paper_{args.dataset_name}.json"), "r") as json_file: # Open sampled paper info
    papers = json.load(json_file)
with open(os.path.join(SAMPLE_DIR, f"review_{args.dataset_name}.json"), "r") as json_file: # Open sampled review info
    reviews = json.load(json_file)

def is_file_larger_than(path,lim): # 10 kb
    """Checks if a file is larger than a given limit in bytes."""

    try:
        size = os.path.getsize(path) # Get the size of respective file
        return size > lim # Check if size > lim
    except FileNotFoundError: # If the file does not exist
        return False

if __name__ == "__main__":

    for i, paper in enumerate(tqdm(papers, desc="Downloading Papers")): # Show download progress

        path_pdf = os.path.join(PDF_DIR, paper["id"] + ".pdf") # Create path for downloaded PDFs using PDF_DIR

        if not is_file_larger_than(path_pdf, 10*1024): # If size <= lim or file is missing
            time.sleep(0.5) # Pause execution for 0.5 sec
            url = "https://openreview.net" + paper["pdf"] # Complete URL for respective PDF
            print(i, url)
            response = requests.get(url, proxies={"http": None, "https": None}) # Request for given PDF

            if response.status_code == 200: # Check request
                with open(path_pdf, "wb") as file:
                    file.write(response.content) # Write downloaded binary content (PDF) to the file
            else:
                print("Failed to download the file. Status code:", response.status_code) # Print error message
                time.sleep(2) # Pause execution for 2 sec
                response = requests.get(url, proxies={"http": None, "https": None}) # Request again for given PDF
                if response.status_code == 200: # Check status
                    with open(path_pdf, "wb") as file:
                        file.write(response.content) # Write downloaded binary content (PDF) to the file
                else:
                    print("Failed again:", response.status_code) # Print a final error message
                    raise(Exception)

    print("\nDownload complete!")

In [ ]:
# @title ZIP PDFs

path_folder = PDF_DIR # Use the defined PDF_DIR
path_zip = os.path.join(path_folder, "paper.zip") # Path of the zip-file

file_paths = [] # Create empty list for full paths of all to be zipped PDFs
for root, _, files in os.walk(path_folder): # Iterate through the directory
    for file in files:
        if file.endswith(".pdf"): # Only add PDF files, exclude the zip itself if it exists
            file_paths.append(os.path.join(root, file)) # The full path is added to the file_paths

if not file_paths:
    print(f"No PDF files found in {path_folder} to zip.")
else:
    with zipfile.ZipFile(path_zip, 'w', zipfile.ZIP_DEFLATED) as zipf: # Open in write mode and use DEFLATED compression
        for file in tqdm(file_paths, desc="Zipping files"):
            zipf.write(file, arcname=os.path.basename(file)) # Add current file to the zip, preserve only filename

    print("\nZIP-file created!")

# Parsing

In [ ]:
# @title [GROBID]

# Define input path (PDF-files) using PDF_DIR
INPUT_DIR = PDF_DIR

# Define output path (TEI XML-files) using GROBID_DIR
OUTPUT_DIR = GROBID_DIR

# Define API-call function
def pdf2grobid(filename, GROBID_API):
    """Sends a single PDF to the GROBID API and returns the TEI XML text."""

    try:
        with open(filename, 'rb') as file:
            files = {'input': file}
            # Long timeout for safety
            response = requests.post(GROBID_API, files=files, timeout=300)

        # Check for response error
        if response.status_code != 200:
            response.raise_for_status()

        # Capture the text of the API response (expected as tei_xml)
        return response.text

    # Handle errors
    except requests.exceptions.RequestException as e:
        print(f"Error processing {filename}: {e}")
        return None

# Iterate over all PDF files in the input directory
for filename in os.listdir(INPUT_DIR):
    if filename.endswith(".pdf"):
        # Construct the full input path
        input_file_path = os.path.join(INPUT_DIR, filename)

        # Replace .pdf with .xml for the output filename
        xml_filename = filename.replace(".pdf", ".xml")

        # Construct the full output path
        output_file_path = os.path.join(OUTPUT_DIR, xml_filename)

        print(f"Processing: {filename}...")

        # Call defined function for parsing
        tei_xml = pdf2grobid(input_file_path, GROBID_API)

        if tei_xml:
            # Write a TEI XML-file into the output path
            with open(output_file_path, 'w', encoding='utf-8') as f:
                f.write(tei_xml)
            print(f"  -> Success: Saved to {xml_filename}")

        # Add short delay for safety
        time.sleep(3)

print("\nProcessing complete!")

In [ ]:
# @title landingAI

# # Failed filenames
# FAILED_FILENAMES = [
#     "wWg_Ee5q_W.pdf",
#     "dmWMfJeZMM.pdf",
#     "UvmDCdSPDOW.pdf",
#     "U7LLhh3VFxH.pdf",
#     "_BoPed4tYww.pdf",
#     "gxq1n1f0c7l.pdf"
# ]

# # Define input path (PDF-files) using PDF_DIR
# INPUT_DIR = PDF_DIR
# # Create input path if not already existing (already done in Paths cell)

# # Define output dir (JSON-files) using LANDINGAI_DIR
# OUTPUT_DIR_JSON = LANDINGAI_DIR
# # Create output path if not already existing (already done in Paths cell)

# # Define the new function for the LandingAI API call
# def pdf_to_landingai_json(filename, url, headers, data):
#     """Sends a single PDF to the LandingAI API and returns the parsed JSON."""

#     # Test if API-Key was placed
#     if LANDING_AI_API_KEY == "YOUR_API_KEY":
#         print("ERROR: Please update LANDING_AI_API_KEY with your actual key.")
#         return None

#     try:
#         # Open the PDF in binary mode for upload
#         with open(filename, 'rb') as document:
#             files = {'document': document}

#             # Send the request
#             # Use timout for safety
#             response = requests.post(url, files=files, data=data, headers=headers, timeout=300)

#         # Check for response error
#         if response.status_code != 200:
#             print(f"\n  -> API Error {response.status_code}: {response.text}")
#             response.raise_for_status()

#         # Capture response as json
#         return response.json()

#     # In case of an error
#     except requests.exceptions.RequestException as e:
#         print(f"\n  -> Connection Error: {e}")
#         return None

# # Print start of the process
# print(f"Starting retry process for {len(FAILED_FILENAMES)} files using LandingAI...")

# for filename in FAILED_FILENAMES:

#     # Construct the full input path
#     input_file_path = os.path.join(INPUT_DIR, filename)

#     # Construct the full output path
#     json_filename = filename.replace(".pdf", ".json")
#     output_file_path = os.path.join(OUTPUT_DIR_JSON, json_filename)

#     # Print current status
#     print(f"Processing: {filename}...")

#     # Call the new function
#     parsed_json_output = pdf_to_landingai_json(
#         filename=input_file_path, url=LANDING_AI_API, headers=LANDING_AI_HEADERS, data=LANDING_AI_MODEL
#     )

#     if parsed_json_output:
#         # Write json-file
#         with open(output_file_path, 'w', encoding='utf-8') as f:
#             # Use indent=4 for readable formatting
#             json.dump(parsed_json_output, f, indent=4)

#         # Message in case of success
#         print(f"  -> Success: Saved JSON to {json_filename}")
#     else:
#         # Message in case of failure
#         print(f"  -> Final failure: Could not process {filename}.")

#     # Add short delay for safety
#     time.sleep(1)

# print("\nProcessing complete!")

# Conversion

In [ ]:
# @title TEI XML to TXT [GROBID]

# Define input directory (where GROBID XML files are stored) using GROBID_DIR
INPUT_DIR = GROBID_DIR

# Define output directory for TXT files using TXT_DIR
OUTPUT_DIR = TXT_DIR
# Create the output path if not already existing (already done in Paths cell)

# Define extraction function
def extract_text_from_tei_xml(xml_content):
    """Parses TEI XML and extracts clean body text."""

    # Use 'lxml' as the parser for speed and robustness with XML
    soup = BeautifulSoup(xml_content, 'lxml')

    body_tag = soup.find('body') # 'body' = main text

    if body_tag:
        # .get_text(separator=' ', strip=True) extracts all nested text
        # And cleanly separates it with spaces, removing excess newlines/whitespace.
        clean_text = body_tag.get_text(separator=' ', strip=True)
        return clean_text
    return "" # Return empty string if body tag is not found

# Iterate across all .xml-files in the INPUT_XMR_DIR
xml_files = [f for f in os.listdir(INPUT_DIR) if f.endswith('.xml')]
# Message how many files to convert
print(f"Found {len(xml_files)} XML files to convert.")

# Message progress of file conversion
for filename in tqdm(xml_files, desc="Converting XML to TXT"):

    # Construct the full input path
    xml_path = os.path.join(INPUT_DIR, filename)

    # Replace .xml with .txt for the output file name
    txt_filename = filename.replace(".xml", ".txt")

    # Construct the full output path
    txt_path = os.path.join(OUTPUT_DIR, txt_filename)

    try:
        # Read the input fill
        with open(xml_path, 'r', encoding='utf-8') as f:
            xml_content = f.read()

        # Call the defined function for extraction
        parsed_text = extract_text_from_tei_xml(xml_content)

        # Write a TXT-file into the output path
        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write(parsed_text)

    # Print error message
    except Exception as e:
        print(f"\nError processing {filename}: {e}")
        continue # Skip to the next file

print("\nConversion complete! Text files are now in the 'txt' folder.")

In [ ]:
# @title JSON to TXT [landingAI]

# Tbd: On what logic should I extract the main text out of the .txt-file?

# Final dataset

In [ ]:
# @title Build dataset

# Ensure args is defined from the 'Sampling' cell (7Y6VDccPoWrW) before running this.
# If running this cell independently, you might need to re-define args, SAMPLE_DIR, and TXT_DIR.

# Open paper_rand2000 from SAMPLE_DIR
with open(os.path.join(SAMPLE_DIR, f"paper_{args.dataset_name}.json"), "r") as json_file:
    papers = json.load(json_file)
# Open review_rand2000 from SAMPLE_DIR
with open(os.path.join(SAMPLE_DIR, f"review_{args.dataset_name}.json"), "r") as json_file:
    reviews = json.load(json_file)

# Iterates through all reviews and maps all of them onto their ID
mp = {}
for review in reviews:
    if review["belong_id"] not in mp:
        mp[review["belong_id"]] = []
    mp[review["belong_id"]].append(review["review"])

# Asserts:
assert(len(mp)==len(papers)) # A complete map of reviews for all selected papers
cnt = 0
for (paper_id, reviews_list) in mp.items(): # Changed 'reviews' to 'reviews_list' to avoid shadowing outer 'reviews'
    assert(len(reviews_list) >= 3) # Sufficient amount reviews
    random.seed(19260817 + cnt)
    cnt += 1
    random.shuffle(reviews_list) # Shuffles the review order

# Create dataframe
df = pd.DataFrame(columns=["paper_id", "pdf_url", "abstract", "parsed_text","human_review1","human_review2","human_review3"]) # With respective columns
for i, paper in enumerate(tqdm(papers, desc="Processing Papers")): # Iterate over the papers
    pdf_url = "https://openreview.net" + paper["pdf"] # Create the papers PDF-url

    # As not all PDFs were responded by GROBID and converted to TXT, this block ensures the code runs anyways!
    try:
        txt_filepath = os.path.join(TXT_DIR, paper["id"] + ".txt") # Use TXT_DIR
        with open(txt_filepath, "r", encoding="utf-8") as f: # Open the respective .txt-file
            parsed_text = f.read() # And read it into parsed_text

    except FileNotFoundError:
        print(f"Skipping paper (Raw text file not found at: {txt_filepath}).")
        continue # Skip this paper if the raw text is missing
    except Exception as e:
        # Watch other potential file reading issues (like encoding errors)
        print(f"Error reading raw text for paper {paper['id']}: {e}") # Use paper['id'] for clarity
        continue

    # Place the respective content at position loc[len(df)]
    df.loc[len(df)] = [paper["id"], pdf_url, paper["abstract"], parsed_text, mp[paper["id"]][0],mp[paper["id"]][1],mp[paper["id"]][2]]

# Save dataset as .parquet
df.to_parquet(os.path.join(DATASET_DIR, 'dataset_paper.parquet'), index=False) # Use os.path.join for DATASET_DIR

print("\nDataset created!")

In [ ]:
# @title Check dataset

# Read dataset_paper
dataset_paper = pd.read_parquet(os.path.join(DATASET_DIR, "dataset_paper.parquet"))

for i in range(len(dataset_paper)):
        row = dataset_paper.loc[i] # Retrive the i-th row
        if len(row["parsed_text"]) < 1000: # Check if length "parsed_text" < 1000 (untypical short)
            print(i, row["paper_id"], len(row["parsed_text"])) # Print index, paper ID and length of "parsed_text"

# Print head
display(dataset_paper.head(10))

print("\nDataset head printed!")

# References

Xu, S., Lu, Y., Schoenebeck, G., & Kong, Y. (2024). Benchmarking LLMs'   Judgments with No Gold Standard. *arXiv preprint arXiv:2411.07127.*

The repository is available [here](https://github.com/yx-lu/Benchmarking-LLMs--Judgments-with-No-Gold-Standard)
